# Dental Implant Survival Analysis – Andersen–Gill Cox Models

**Target journal:** Journal of Clinical Periodontology (JCP)

This notebook performs a comprehensive survival analysis of dental implants using
Andersen–Gill (AG) Cox proportional hazards models with robust standard errors
(clustered by patient). Three models are fitted:

1. **Overall AG model** – full cohort
2. **Early-failure AG model** – failures within ≤365 days
3. **Late-failure AG model** – failures after >365 days

The analysis also includes Kaplan–Meier curves with time-at-risk tables,
expanded univariable survival screening, and a global proportional-hazards
diagnostic for each AG model.

All functions are imported from `functions.py` for modularity and reproducibility.

In [ ]:
# ============================================================
# 0. Setup & Imports
# ============================================================
import importlib
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project module
import functions as fn
importlib.reload(fn)

fn.print_versions()

# Display settings
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## 1. Configuration

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
INPUT_PATH = r"C:\Users\cahana_s\research\research_shetel_2_3\shetel_2_3_data\implants_with_restoration_summary_all.xlsx"
OUTDIR     = r"C:\Users\cahana_s\research\research_shetel_2_3\outputs"
CENSOR_DATE   = "2025-08-24"
COHORT_CUTOFF = "2023-07-01"
EARLY_THRESHOLD_DAYS = 365   # ≤365 = early, >365 = late

import os
os.makedirs(OUTDIR, exist_ok=True)

## 2. Data Loading & Preprocessing

In [ ]:
# ============================================================
# 2. Data Loading & Preprocessing
# ============================================================
from pathlib import Path
import pandas as pd

PARQUET_PATH = Path("final_cohort.parquet")

if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
else:
    df_raw = fn.load_data(INPUT_PATH)
    df = fn.preprocess(
        df_raw,
        censor_date=CENSOR_DATE,
        cohort_cutoff=COHORT_CUTOFF
    )

    object_cols = df.select_dtypes(include=["object"]).columns.tolist()
    print("Casting object columns to string:", object_cols)

    for col in object_cols:
        df[col] = df[col].astype("string").str.strip()

    df.to_parquet(PARQUET_PATH, index=False)

print(f"Final cohort: {len(df):,} implants | {df['patient_id'].nunique():,} patients")
print(f"Failures:     {int(df['is_failure'].sum()):,} ({df['is_failure'].mean()*100:.1f}%)")

## 3. Descriptive Statistics

In [ ]:
# ============================================================
# 3a. Continuous variables
# ============================================================
continuous_cols = ["age_years", "length", "diameter", "days_to_failure"]
display(fn.style_table(
    fn.continuous_summary(df, continuous_cols),
    caption="Continuous Variables"
))

In [ ]:
# ============================================================
# 3b. Categorical / binary variables
# ============================================================
cat_cols = [
    "jaw", "diameter_cat", "length_cat", "age_bin",
    "gender_bin", "implant_num_surv_lbl",
    "rehabilitation_type", "fixed_loading_type",
    "smoker", "has_diabetes", "has_hypertension",
    "takes_biphos", "Penicillin_Allergy",
    "has_bonegraft_beforeimplant", "has_rama_onimplantday",
    "has_mahash_onimplantday", "has_resm_onimplantday",
    "has_resp_onorbeforeimplant",
]
display(fn.style_table(
    fn.frequency_table(df, cat_cols),
    caption="Categorical and Binary Variables"
))

In [ ]:
# ============================================================
# 3c. Follow-up summary
# ============================================================
display(fn.style_table(
    fn.follow_up_summary(df),
    caption="Follow-up Summary"
))

## 4. Implant Survival by Procedure Sequence

In [ ]:
# ============================================================
# 4. Stacked bar chart – survival vs failure by implant sequence
# ============================================================
fig = fn.plot_survival_by_sequence(df, save_path=os.path.join(OUTDIR, "survival_by_sequence"))
plt.show()

## 5. Kaplan-Meier Survival and Follow-up Completeness

The two panels are displayed side by side. Panel A shows potential follow-up using reverse Kaplan-Meier, with group medians marked directly on the plot. The time-at-risk table for panel B is printed separately below the figure.

**Panel A:** reverse Kaplan-Meier follow-up by implant sequence (1, 2, 3+).

**Panel B:** Kaplan-Meier survival by implant sequence (1, 2, 3+).

For panel A, follow-up is censored administratively at 2023-12-31.

In [ ]:
# ============================================================
# 5. Composite figure: panel A reverse KM; panel B KM
# ============================================================
df["years_to_failure"] = df["days_to_failure"] / 365.25
df["implant_group"] = df["implant_num_surv"].apply(lambda g: "3+" if g >= 3 else str(int(g)))

label_map = {"1": "First implant", "2": "Second implant", "3+": "Third or more"}
color_map = {"1": fn.WONG["blue"], "2": fn.WONG["orange"], "3+": fn.WONG["green"]}

fig = fn.plot_implant_sequence_survival_followup_figure(
    df,
    save_path=os.path.join(OUTDIR, "km_reverse_km_implant_sequence"),
    study_end="2023-12-31",
    duration_col="years_to_failure",
    event_col="is_failure",
    group_col="implant_group",
    label_map=label_map,
    color_map=color_map,
)
plt.show()

km_risk_table = fn.km_time_at_risk_table(
    df,
    duration_col="years_to_failure",
    event_col="is_failure",
    group_col="implant_group",
    label_map=label_map,
    x_cap=10.0,
)
print("Time at risk for panel B:")
display(fn.style_table(km_risk_table))

reverse_km_summary = fn.reverse_km_followup_summary(
    df,
    study_end="2023-12-31",
    group_col="implant_group",
    event_col="is_failure",
    label_map=label_map,
)
print("Potential follow-up summary for panel A:")
display(fn.style_table(reverse_km_summary))

In [ ]:
# ============================================================
# 6. Empirical check: follow-up starts at each site's own implant date
# ============================================================
study_end_panel_a = pd.Timestamp("2023-12-31")
study_end_panel_b = pd.Timestamp(CENSOR_DATE)

check = df.copy()
check["patient_id"] = check["patient_id"].astype("string")
check["tooth_number"] = pd.to_numeric(check["tooth_number"], errors="coerce")
check["site_id"] = (
    check["patient_id"].fillna("NA")
    + "_"
    + check["tooth_number"].astype("Int64").astype("string").fillna("NA")
)

check["duedate"] = pd.to_datetime(check["duedate"], errors="coerce")
check["failure_date"] = pd.to_datetime(check["failure_date"], errors="coerce")
check["is_failure"] = pd.to_numeric(check["is_failure"], errors="coerce").fillna(0).astype(int)
check["implant_index"] = pd.to_numeric(check["implant_index"], errors="coerce")
check["implant_num_surv"] = pd.to_numeric(check["implant_num_surv"], errors="coerce")

check["panel_a_end"] = check["failure_date"].where(check["is_failure"].eq(1), study_end_panel_a)
check["panel_b_end"] = check["failure_date"].where(check["is_failure"].eq(1), study_end_panel_b)

check["panel_a_followup_days"] = (check["panel_a_end"] - check["duedate"]).dt.days
check["panel_b_followup_days"] = (check["panel_b_end"] - check["duedate"]).dt.days

sites_3plus = (
    check.groupby("site_id")["implant_index"]
    .max()
    .loc[lambda s: s >= 3]
    .index
)

subset = (
    check.loc[check["site_id"].isin(sites_3plus), [
        "site_id",
        "patient_id",
        "tooth_number",
        "implant_index",
        "implant_num_surv",
        "duedate",
        "failure_date",
        "is_failure",
        "panel_a_end",
        "panel_a_followup_days",
        "panel_b_end",
        "panel_b_followup_days",
    ]]
    .sort_values(["site_id", "implant_index", "duedate"])
    .reset_index(drop=True)
)

summary_check = pd.DataFrame([{
    "Sites with >=3 implants": subset["site_id"].nunique(),
    "Patients represented": subset["patient_id"].nunique(),
    "Rows with implant_index == 2": int((subset["implant_index"] == 2).sum()),
    "Rows with implant_index >= 3": int((subset["implant_index"] >= 3).sum()),
    "All implant 2 rows have own start date": subset.loc[subset["implant_index"] == 2, "duedate"].notna().all(),
    "All implant 3+ rows have own start date": subset.loc[subset["implant_index"] >= 3, "duedate"].notna().all(),
}])

print("Empirical check by site_id (same patient + same tooth):")
display(fn.style_table(summary_check))

print("Example rows for sites with 3+ implants:")
display(fn.style_table(subset.head(50)))

In [ ]:
# ============================================================
# 6b. Composite figure: reverse KM excludes early failures; KM keeps full data
# ============================================================
early_failure_mask = (
    pd.to_numeric(df["is_failure"], errors="coerce").fillna(0).astype(int).eq(1)
    & pd.to_numeric(df["days_to_failure"], errors="coerce").notna()
    & (pd.to_numeric(df["days_to_failure"], errors="coerce") <= EARLY_THRESHOLD_DAYS)
)

df_reverse_no_early = df.loc[~early_failure_mask].copy()
df_reverse_no_early["years_to_failure"] = df_reverse_no_early["days_to_failure"] / 365.25
df_reverse_no_early["implant_group"] = df_reverse_no_early["implant_num_surv"].apply(
    lambda g: "3+" if g >= 3 else str(int(g))
)

df_km_full = df.copy()
df_km_full["years_to_failure"] = df_km_full["days_to_failure"] / 365.25
df_km_full["implant_group"] = df_km_full["implant_num_surv"].apply(
    lambda g: "3+" if g >= 3 else str(int(g))
)

label_map_early = {"1": "First implant", "2": "Second implant", "3+": "Third or more"}
color_map_early = {"1": fn.WONG["blue"], "2": fn.WONG["orange"], "3+": fn.WONG["green"]}

print(
    f"Reverse KM subset after excluding early failures (<= {EARLY_THRESHOLD_DAYS} days): "
    f"{len(df_reverse_no_early):,} implants"
)
print(f"Regular KM subset (full data): {len(df_km_full):,} implants")

fn._apply_jcp_style()
fig_early_failure = plt.figure(figsize=(13.2, 5.4), constrained_layout=True)
gs = fig_early_failure.add_gridspec(
    nrows=1,
    ncols=2,
    width_ratios=[1.1, 1.0],
    wspace=0.18,
)

ax_a = fig_early_failure.add_subplot(gs[0, 0])
ax_b = fig_early_failure.add_subplot(gs[0, 1])

combined_reverse = fn.compute_followup_time(
    df_reverse_no_early,
    study_end="2023-12-31",
    event_col="is_failure",
)
reverse_summary_early = fn.reverse_km_followup_summary(
    df_reverse_no_early,
    study_end="2023-12-31",
    group_col="implant_group",
    event_col="is_failure",
    label_map=label_map_early,
)

fn._plot_survival_curves_on_axis(
    combined_reverse,
    ax=ax_a,
    duration_col="followup_years",
    event_col="reverse_km_event",
    group_col="implant_group",
    label_map=label_map_early,
    color_map=color_map_early,
    title="Reverse Kaplan-Meier follow-up by implant sequence\n(excluding failures <= 1 year)",
    ylabel="Probability of remaining under follow-up",
    reverse=True,
    x_cap=10.0,
)
fn._annotate_reverse_km_medians(ax_a, reverse_summary_early, label_map_early, color_map_early)
ax_a.text(
    -0.12,
    1.04,
    "A",
    transform=ax_a.transAxes,
    ha="left",
    va="top",
    fontsize=12,
    fontweight="bold",
    color=fn.FIG_DEFAULTS["title_color"],
    clip_on=False,
)

fn._plot_survival_curves_on_axis(
    df_km_full,
    ax=ax_b,
    duration_col="years_to_failure",
    event_col="is_failure",
    group_col="implant_group",
    label_map=label_map_early,
    color_map=color_map_early,
    title="Kaplan-Meier survival by implant sequence",
    ylabel="Survival probability",
    reverse=False,
    x_cap=10.0,
)
ax_b.text(
    -0.12,
    1.04,
    "B",
    transform=ax_b.transAxes,
    ha="left",
    va="top",
    fontsize=12,
    fontweight="bold",
    color=fn.FIG_DEFAULTS["title_color"],
    clip_on=False,
)

save_path_early = os.path.join(
    OUTDIR,
    "km_reverse_km_implant_sequence_reverse_excluding_early_failures",
)
for ext in ("png", "pdf"):
    fig_early_failure.savefig(
        f"{save_path_early}.{ext}",
        dpi=fn.FIG_DEFAULTS["dpi"],
        bbox_inches="tight",
    )
print(f"  Saved: {save_path_early}.png / .pdf")
plt.show()

km_risk_table_early = fn.km_time_at_risk_table(
    df_km_full,
    duration_col="years_to_failure",
    event_col="is_failure",
    group_col="implant_group",
    label_map=label_map_early,
    x_cap=10.0,
)
print("Time at risk for panel B (full data):")
display(fn.style_table(km_risk_table_early))

print("Potential follow-up summary for panel A (excluding failures <= 1 year):")
display(fn.style_table(reverse_summary_early))

In [ ]:
# ============================================================
# 6c. Follow-up time distribution by implant index group
# ============================================================
study_end_followup = "2023-12-31"
followup_plot = fn.compute_followup_time(df, study_end=study_end_followup, event_col="is_failure")

group_source_col = "implant_index" if "implant_index" in followup_plot.columns else "implant_num_surv"
group_num = pd.to_numeric(followup_plot[group_source_col], errors="coerce")
followup_plot["implant_group"] = pd.Series(pd.NA, index=followup_plot.index, dtype="string")
followup_plot.loc[group_num.eq(1), "implant_group"] = "1"
followup_plot.loc[group_num.eq(2), "implant_group"] = "2"
followup_plot.loc[group_num.ge(3), "implant_group"] = "3+"
followup_plot = followup_plot.loc[followup_plot["implant_group"].isin(["1", "2", "3+"])].copy()

followup_bin_summary = fn.summarize_followup_bins(
    followup_plot,
    group_col="implant_group",
    followup_col="followup_years",
    allowed_groups=["1", "2", "3+"],
)

label_map = {"1": "First implant", "2": "Second implant", "3+": "Third or more"}
color_map = {"1": fn.WONG["blue"], "2": fn.WONG["orange"], "3+": fn.WONG["green"]}

fn._apply_jcp_style()
fig_followup_dist, ax_followup_dist = plt.subplots(figsize=(8.8, 4.8), constrained_layout=True)
fn.plot_followup_bin_summary(
    followup_bin_summary,
    ax=ax_followup_dist,
    label_map=label_map,
    color_map=color_map,
    title="Follow-up time distribution by implant index group",
)
ax_followup_dist.set_xlabel("Follow-up time category")

save_path_followup_dist = os.path.join(OUTDIR, "followup_time_distribution_by_implant_index_group")
for ext in ("png", "pdf"):
    fig_followup_dist.savefig(f"{save_path_followup_dist}.{ext}", dpi=fn.FIG_DEFAULTS["dpi"], bbox_inches="tight")
print(f"  Saved: {save_path_followup_dist}.png / .pdf")
plt.show()

In [ ]:
# ============================================================
# 6d. Follow-up time distribution by implant index group (percent)
# ============================================================
percent_bin_order = ["<=1 year", ">1-3 years", ">3-5 years", ">5 years"]
percent_group_order = ["1", "2", "3+"]

followup_percent_plot = (
    followup_bin_summary
    .assign(
        implant_group=followup_bin_summary["implant_group"].astype(str),
        followup_bin=followup_bin_summary["followup_bin"].astype(str),
    )
    .pivot(index="implant_group", columns="followup_bin", values="percent")
    .reindex(index=percent_group_order, columns=percent_bin_order)
    .fillna(0.0)
)

percent_colors = {
    "<=1 year": "#C44E52",
    ">1-3 years": "#DD8452",
    ">3-5 years": "#55A868",
    ">5 years": "#4C72B0",
}
percent_labels = {
    "1": "First implant",
    "2": "Second implant",
    "3+": "Third or more",
}

fn._apply_jcp_style()
fig_followup_pct, ax_followup_pct = plt.subplots(figsize=(8.6, 5.0), constrained_layout=True)

bottom = np.zeros(len(followup_percent_plot.index), dtype=float)
x = np.arange(len(followup_percent_plot.index))

for bin_name in percent_bin_order:
    values = followup_percent_plot[bin_name].to_numpy(dtype=float)
    bars = ax_followup_pct.bar(
        x,
        values,
        bottom=bottom,
        color=percent_colors[bin_name],
        edgecolor="white",
        linewidth=0.8,
        label=bin_name,
    )
    for bar, value, base in zip(bars, values, bottom):
        if value >= 6:
            ax_followup_pct.text(
                bar.get_x() + bar.get_width() / 2,
                base + value / 2,
                f"{value:.1f}%",
                ha="center",
                va="center",
                color="white",
                fontsize=8,
                fontweight="bold",
            )
    bottom = bottom + values

ax_followup_pct.set_xticks(x)
ax_followup_pct.set_xticklabels([percent_labels.get(idx, idx) for idx in followup_percent_plot.index])
ax_followup_pct.set_ylabel("Percent of implants")
ax_followup_pct.set_xlabel("Implant index group")
ax_followup_pct.set_ylim(0, 100)
ax_followup_pct.set_title("Follow-up time distribution by implant index group (%)", color=fn.FIG_DEFAULTS["title_color"], pad=8)
ax_followup_pct.grid(axis="y", color=fn.FIG_DEFAULTS["grid_color"], alpha=0.6)
ax_followup_pct.legend(
    title="Follow-up time",
    loc="upper center",
    bbox_to_anchor=(0.5, 1.14),
    ncol=4,
    framealpha=0.95,
    facecolor="white",
    edgecolor=fn.FIG_DEFAULTS["border_color"],
)
for spine in ["top", "right"]:
    ax_followup_pct.spines[spine].set_visible(False)

save_path_followup_pct = os.path.join(OUTDIR, "followup_time_distribution_by_implant_index_group_percent")
for ext in ("png", "pdf"):
    fig_followup_pct.savefig(f"{save_path_followup_pct}.{ext}", dpi=fn.FIG_DEFAULTS["dpi"], bbox_inches="tight")
print(f"  Saved: {save_path_followup_pct}.png / .pdf")
plt.show()

## 6. Univariable Survival Screening

In [ ]:
# ============================================================
# 6. Log-rank plus univariable Cox screening
# ============================================================
test_vars = [
    "jaw", "diameter_cat", "length_cat", "age_bin",
    "gender_bin", "implant_num_surv",
    "smoker", "has_diabetes", "has_hypertension",
    "takes_biphos", "Penicillin_Allergy",
    # "has_bonegraft_beforeimplant", "has_rama_onimplantday",
    # "has_mahash_onimplantday", "has_resm_onimplantday",
    # "has_resp_onorbeforeimplant",
]
lr_results = fn.logrank_all_variables(df, test_vars)
univ_results = fn.univariable_survival_summary(df, test_vars)
univ_results_pub = fn.make_univariable_publication_table(univ_results)

print("Univariable survival screening:")
display(fn.style_univariable_table(univ_results))
print("\nManuscript-ready univariable table:")
display(fn.style_univariable_publication_table(univ_results_pub))
print(f"\nSignificant log-rank tests (p < 0.05): {int(lr_results['Significant'].sum())} / {len(lr_results)}")

## 7. Cox Model Data Preparation

In [ ]:
# ============================================================
# 7. Prepare the model frame for AG models
# ============================================================
df = fn.prepare_cox_time(df, study_end=CENSOR_DATE)
mf = fn.prepare_model_frame(df)

print(f"Model frame: {len(mf):,} rows")
print(f"Failure events: {int(mf['failure_event'].sum()):,}")

## 8. Andersen–Gill Cox Model – Overall Cohort

This model includes the **full cohort** (all implant failures regardless of timing).
Each patient may contribute multiple implants (AG framework) with robust SE via `cluster(patient_id)`.

In [ ]:
# ============================================================
# 8. AG Model – Overall
# ============================================================
dat_overall = fn.prepare_ag_data(mf)

res_overall, n_overall, ev_overall, c_overall, ph_overall = fn.run_ag_model_r(
    dat_overall, model_label="Andersen–Gill Cox Model – Overall"
)

tbl_overall = fn.result_table(
    res_overall, n_overall, ev_overall, c_overall, ph_overall,
    model_label="AG Model – Overall"
 )
tbl_overall_pub = fn.result_table_publication(
    res_overall, n_overall, ev_overall, c_overall, ph_overall,
    model_label="AG Model – Overall"
)
diag_overall = fn.model_diagnostics_table(
    "AG Model – Overall", n_overall, ev_overall, c_overall, ph_overall
)
display(fn.style_comparison_table(diag_overall))
display(fn.style_result_table(tbl_overall))
print("Manuscript-ready Cox table:")
display(fn.style_result_table(tbl_overall_pub))

In [ ]:
# Forest plot – Overall
fig_overall = fn.forest_plot(
    res_overall, n_overall, ev_overall, c_overall,
    model_label="Andersen–Gill Cox Model – Overall",
    save_path=os.path.join(OUTDIR, "forest_AG_overall"),
)
plt.show()

fig_overall_journal = fn.forest_plot_journal(
    res_overall, n_overall, ev_overall, c_overall,
    model_label="Andersen–Gill Cox Model – Overall",
    save_path=os.path.join(OUTDIR, "forest_AG_overall_journal"),
)
plt.show()

## 9. Early / Late Failure Classification

**Definition (from the original analysis):**
- **Early failure:** implant failure within ≤365 days of placement
- **Late failure:** implant failure after >365 days
- **Censored:** no failure observed during follow-up

In [ ]:
# ============================================================
# 9. Classify failures as early / late
# ============================================================
mf = fn.classify_failure_type(mf, threshold_days=EARLY_THRESHOLD_DAYS)

print("Failure type distribution:")
print(mf["failure_type"].value_counts(dropna=False))
print(f"\nEarly threshold: ≤{EARLY_THRESHOLD_DAYS} days")

## 10. Andersen–Gill Cox Model – Early Failures (≤365 days)

For this sub-model, only **early failures** (≤365 days) are treated as events.
Late failures and censored observations are treated as censored.

In [ ]:
# ============================================================
# 10. AG Model – Early Failures
# ============================================================
mf_early = mf.copy()
# Re-code: only early failures count as events
mf_early["failure_event"] = np.where(mf_early["failure_type"] == "Early", 1, 0).astype(int)

dat_early = fn.prepare_ag_data(mf_early)

res_early, n_early, ev_early, c_early, ph_early = fn.run_ag_model_r(
    dat_early, model_label="AG Cox Model – Early Failures (≤365 d)"
)

tbl_early = fn.result_table(
    res_early, n_early, ev_early, c_early, ph_early,
    model_label="AG Model – Early Failures"
 )
tbl_early_pub = fn.result_table_publication(
    res_early, n_early, ev_early, c_early, ph_early,
    model_label="AG Model – Early Failures"
)
diag_early = fn.model_diagnostics_table(
    "AG Model – Early Failures", n_early, ev_early, c_early, ph_early
)
display(fn.style_comparison_table(diag_early))
display(fn.style_result_table(tbl_early))
print("Manuscript-ready Cox table:")
display(fn.style_result_table(tbl_early_pub))

In [ ]:
# Forest plot – Early
fig_early = fn.forest_plot(
    res_early, n_early, ev_early, c_early,
    model_label="AG Cox Model – Early Failures (≤365 d)",
    save_path=os.path.join(OUTDIR, "forest_AG_early"),
)
plt.show()

fig_early_journal = fn.forest_plot_journal(
    res_early, n_early, ev_early, c_early,
    model_label="AG Cox Model – Early Failures (≤365 d)",
    save_path=os.path.join(OUTDIR, "forest_AG_early_journal"),
)
plt.show()

## 11. Andersen–Gill Cox Model – Late Failures (>365 days)

For this sub-model, only **late failures** (>365 days) are treated as events.
Early failures and censored observations are treated as censored.

In [ ]:
# ============================================================
# 11. AG Model – Late Failures
# ============================================================
mf_late = mf.copy()
# Re-code: only late failures count as events
mf_late["failure_event"] = np.where(mf_late["failure_type"] == "Late", 1, 0).astype(int)

dat_late = fn.prepare_ag_data(mf_late)

res_late, n_late, ev_late, c_late, ph_late = fn.run_ag_model_r(
    dat_late, model_label="AG Cox Model – Late Failures (>365 d)"
)

tbl_late = fn.result_table(
    res_late, n_late, ev_late, c_late, ph_late,
    model_label="AG Model – Late Failures"
 )
tbl_late_pub = fn.result_table_publication(
    res_late, n_late, ev_late, c_late, ph_late,
    model_label="AG Model – Late Failures"
)
diag_late = fn.model_diagnostics_table(
    "AG Model – Late Failures", n_late, ev_late, c_late, ph_late
)
display(fn.style_comparison_table(diag_late))
display(fn.style_result_table(tbl_late))
print("Manuscript-ready Cox table:")
display(fn.style_result_table(tbl_late_pub))

In [ ]:
# Forest plot – Late
fig_late = fn.forest_plot(
    res_late, n_late, ev_late, c_late,
    model_label="AG Cox Model – Late Failures (>365 d)",
    save_path=os.path.join(OUTDIR, "forest_AG_late"),
)
plt.show()

fig_late_journal = fn.forest_plot_journal(
    res_late, n_late, ev_late, c_late,
    model_label="AG Cox Model – Late Failures (>365 d)",
    save_path=os.path.join(OUTDIR, "forest_AG_late_journal"),
)
plt.show()

## 12. Model Comparison Summary and Diagnostics

In [ ]:
# ============================================================
# 12. Side-by-side model comparison
# ============================================================
comparison = pd.concat([diag_overall, diag_early, diag_late], ignore_index=True)
display(fn.style_comparison_table(comparison))

# Merge HR tables side-by-side for the three models
merged = (
    res_overall[["label", "HR_str", "p_fmt"]].rename(columns={"HR_str": "HR (Overall)", "p_fmt": "p (Overall)"})
    .merge(
        res_early[["label", "HR_str", "p_fmt"]].rename(columns={"HR_str": "HR (Early)", "p_fmt": "p (Early)"}),
        on="label", how="outer"
    )
    .merge(
        res_late[["label", "HR_str", "p_fmt"]].rename(columns={"HR_str": "HR (Late)", "p_fmt": "p (Late)"}),
        on="label", how="outer"
    )
)
merged = merged.rename(columns={"label": "Variable"})
display(fn.style_comparison_table(merged))

## 13. Export Results

In [ ]:
# ============================================================
# 13. Save tables to Excel
# ============================================================
excel_path = os.path.join(OUTDIR, "AG_model_results.xlsx")

with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    tbl_overall.to_excel(writer, sheet_name="Overall", index=False)
    tbl_early.to_excel(writer, sheet_name="Early", index=False)
    tbl_late.to_excel(writer, sheet_name="Late", index=False)
    merged.to_excel(writer, sheet_name="Comparison", index=False)
    comparison.to_excel(writer, sheet_name="Summary", index=False)
    univ_results.to_excel(writer, sheet_name="Univariable", index=False)
    lr_results.to_excel(writer, sheet_name="LogRank", index=False)

    tbl_overall_pub.to_excel(writer, sheet_name="Overall_pub", index=False)
    tbl_early_pub.to_excel(writer, sheet_name="Early_pub", index=False)
    tbl_late_pub.to_excel(writer, sheet_name="Late_pub", index=False)
    univ_results_pub.to_excel(writer, sheet_name="Univariable_pub", index=False)

print(f"Results saved to: {excel_path}")
print(f"Forest plots saved to: {OUTDIR}")
print("\nDone. All outputs ready for JCP submission.")